# Predictive Models: Decision Tree, Neural Network and KNN

**Objective:** Compare Decision Tree, Neural Network, and K-Nearest Neighbor predictive models.

**Dataset:** Breast cancer classification dataset

This notebook is Colab-ready and saves tables, metrics, and visual outputs under
`results/`. Public datasets or compact sample datasets are used so the workflow
remains reproducible.


In [ ]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)


In [ ]:
data = load_breast_cancer(as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.25, random_state=42, stratify=data.target)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Neural Network": MLPClassifier(hidden_layer_sizes=(32,), max_iter=800, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
}

rows = []
predictions = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)
    predictions[name] = pred
    rows.append(
        {
            "model": name,
            "accuracy": accuracy_score(y_test, pred),
            "precision": precision_score(y_test, pred),
            "recall": recall_score(y_test, pred),
            "f1": f1_score(y_test, pred),
        }
    )

metrics = pd.DataFrame(rows).sort_values("f1", ascending=False)
metrics.to_csv(RESULTS_DIR / "model_comparison.csv", index=False)
display(metrics)


In [ ]:
best_model = metrics.iloc[0]["model"]
cm = confusion_matrix(y_test, predictions[best_model])
pd.DataFrame(cm).to_csv(RESULTS_DIR / "best_model_confusion_matrix.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
metrics.set_index("model")[["accuracy", "f1"]].plot(kind="bar", ax=axes[0])
axes[0].set_ylim(0, 1)
axes[0].set_title("Model Comparison")
sns.heatmap(cm, annot=True, fmt="d", cmap="YlGnBu", ax=axes[1])
axes[1].set_title(f"Best Model Confusion Matrix: {best_model}")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "predictive_models_dashboard.png", dpi=180)
plt.show()
